# Orange – Algorytmy AI
## Zadanie szkoleniowe: Conv1D do regresji
### Prosta sieć neuronowa z warstwą Conv1D na zbiorze Pima Indians Diabetes

**Motyw przewodni.** Zbiór Pima zwykle służy do klasyfikacji. Tu robimy świadomy skręt dydaktyczny:
traktujemy dane jako problem **regresji** i przewidujemy poziom **glukozy** na podstawie pozostałych cech.
Conv1D jest narzędziem do pokazania, jak sieć uczy się lokalnych wzorców między uporządkowanymi cechami tabelarycznymi.

| Element | Opis |
|---|---|
| Czas | 60–75 minut |
| Poziom | Podstawowy plus: Python, pandas, ogólna idea sieci neuronowej |
| Biblioteki | pandas, numpy, scikit-learn, tensorflow/keras, matplotlib |
| Typ problemu | Regresja – predykcja wartości ciągłej |
| Architektura | Conv1D + pooling + warstwy Dense |
| Efekt końcowy | Działający notebook, metryki MAE/RMSE, krótka interpretacja |

### Jak pracować z tym notebookiem
- Uzupełniasz komórki oznaczone **`TODO`** – pozostałe komórki są gotowe, uruchamiasz je po kolei.
- Po każdym `TODO` jest komórka ** Sprawdzenie** – jeśli przejdzie bez błędu, możesz iść dalej.
- Plik z danymi `pima.csv` powinien leżeć w tym samym katalogu co notebook.

> **Uwaga etyczna i metodologiczna.** Wyniku nie interpretujemy jako diagnozy. Dane medyczne wymagają walidacji
> klinicznej, kontroli biasu, zgód, bezpieczeństwa i audytu. Budujemy miniaturowy poligon treningowy,
> nie elektronicznego lekarza w pudełku.

## 1. Cel ćwiczenia
- Zrozumieć, jak przygotować dane tabelaryczne do wejścia warstwy Conv1D.
- Zbudować prostą sieć neuronową do regresji w Keras/TensorFlow.
- Porównać błąd modelu na danych treningowych i testowych.
- Zobaczyć, że Conv1D nie jest zarezerwowane dla tekstu i szeregów czasowych – choć w tabeli trzeba ostrożnie interpretować „sąsiedztwo” cech.
- Ocenić model regresyjny za pomocą MAE, MSE, RMSE i wykresu wartości rzeczywistych kontra przewidywanych.

## 2. Importy i ziarno losowości

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras import layers, models

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)

## 3. Dane wejściowe

Zbiór Pima Indians Diabetes: **768 obserwacji, 9 kolumn**. Zmienną docelową jest `Glucose` (wartość ciągła), a nie `Outcome`.

| Kolumna | Znaczenie w ćwiczeniu |
|---|---|
| Pregnancies | Liczba ciąż – cecha wejściowa |
| **Glucose** | **Poziom glukozy – zmienna docelowa regresji** |
| BloodPressure | Ciśnienie krwi – cecha wejściowa |
| SkinThickness | Grubość fałdu skórnego – cecha wejściowa |
| Insulin | Poziom insuliny – cecha wejściowa |
| BMI | Wskaźnik masy ciała – cecha wejściowa |
| DiabetesPedigreeFunction | Funkcja obciążenia rodzinnego – cecha wejściowa |
| Age | Wiek – cecha wejściowa |
| Outcome | Etykieta klasyfikacyjna – w wersji podstawowej pomijamy |

### Krok 1. Wczytanie danych

In [ ]:
columns = [
    "Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
    "Insulin", "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"
]

In [ ]:
# TODO 1: wczytaj dane z pliku pima.csv do zmiennej df i nadaj kolumnom nazwy z listy `columns`.
# Wskazówka: pd.read_csv(...) – sprawdź parametry header i names.
# Jeśli Twój plik ma już wiersz nagłówka (np. wersja z Kaggle), wystarczy header=0, names=columns.

df = None  # <- zastąp tę linię

** Sprawdzenie**

In [ ]:
assert df is not None, "Zmienna df jest pusta – uzupełnij TODO 1."
assert df.shape == (768, 9), f"Oczekiwano (768, 9), jest {df.shape}. Sprawdź parametr header."
assert list(df.columns) == columns, "Nazwy kolumn nie zgadzają się z listą `columns`."
print("OK – dane wczytane poprawnie.")

### Krok 2. Rozmiar, typy kolumn i podstawowe statystyki

In [ ]:
print(df.shape)
display(df.head())
print(df.dtypes)
display(df.describe().T)

### Krok 3. Zera jako braki danych

W kolumnach `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI` wartość **0 jest medycznie nielogiczna** – traktujemy ją jako brak danych.

In [ ]:
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

print("Liczba zer przed zamianą:")
print((df[zero_as_missing] == 0).sum())

df[zero_as_missing] = df[zero_as_missing].replace(0, np.nan)

print("\nLiczba braków (NaN) po zamianie:")
print(df.isna().sum())

### Kroki 4–6. Zmienna docelowa, macierz cech, podział train/test

- Usuwamy wiersze bez wartości docelowej `Glucose`.
- Braki w cechach wejściowych uzupełnimy **medianą liczoną tylko na zbiorze treningowym** (w kolejnym kroku – w pipeline).
- Podział 80/20.

In [ ]:
target_col = "Glucose"
feature_cols = [
    "Pregnancies", "BloodPressure", "SkinThickness", "Insulin",
    "BMI", "DiabetesPedigreeFunction", "Age"
]

df_model = df.dropna(subset=[target_col]).copy()
print("Wiersze po usunięciu braków w Glucose:", len(df_model))

X = df_model[feature_cols]
y = df_model[target_col].to_numpy().reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
print("Train:", X_train.shape, "Test:", X_test.shape)

### Krok 7. Imputacja i skalowanie

Pipeline dla X: imputacja medianą + standaryzacja. Uczymy go (`fit`) **tylko na zbiorze treningowym** – to chroni przed wyciekiem informacji ze zbioru testowego.
Skalujemy też `y`, żeby sieć uczyła się stabilniej.

In [ ]:
x_preprocess = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
X_train_scaled = x_preprocess.fit_transform(X_train)
X_test_scaled = x_preprocess.transform(X_test)

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train)
y_test_scaled = y_scaler.transform(y_test)

print("Średnia y_train_scaled:", round(float(y_train_scaled.mean()), 4),
      "| odchylenie:", round(float(y_train_scaled.std()), 4))

### Krok 8. Tensor 3D dla Conv1D

Conv1D oczekuje kształtu **(próbki, kroki, kanały)**. Każda cecha staje się kolejnym „krokiem” krótkiej sekwencji, a kanał jest jeden.

In [ ]:
X_train_conv = X_train_scaled[..., np.newaxis]
X_test_conv = X_test_scaled[..., np.newaxis]

print("X_train_conv shape:", X_train_conv.shape)
print("X_test_conv shape:", X_test_conv.shape)

## 4. Architektura sieci

| Warstwa | Rola |
|---|---|
| Input | Tensor o kształcie (liczba cech, 1) – każda cecha to kolejny krok sekwencji |
| Conv1D | Małe lokalne filtry; `kernel_size=2` patrzy na pary sąsiednich cech |
| Conv1D | Druga warstwa buduje bardziej złożone reprezentacje |
| GlobalAveragePooling1D | Redukuje wymiar sekwencyjny do wektora |
| Dense | Łączy reprezentacje, transformacja nieliniowa |
| Dense(1) | Jedna liczba: przewidywana `Glucose` na skali standaryzowanej |

> **Uwaga.** Conv1D zakłada lokalność – sąsiednie pozycje mają znaczenie. W szeregach czasowych to naturalne,
> w tabeli medycznej kolejność cech jest **sztuczna**. Dlatego to ćwiczenie uczy mechaniki Conv1D,
> a w realnym projekcie trzeba je porównać z modelem Dense, Random Forest lub Gradient Boosting.

### Krok 9a. Budowa i kompilacja modelu

In [ ]:
# TODO 2: zbuduj model Conv1D i zapisz go w zmiennej model.
# Sugerowana architektura (models.Sequential):
#   - layers.Input(shape=(X_train_conv.shape[1], 1))
#   - layers.Conv1D(filters=..., kernel_size=2, activation="relu")
#   - layers.Conv1D(filters=..., kernel_size=2, activation="relu")
#   - layers.GlobalAveragePooling1D()
#   - layers.Dense(..., activation="relu")
#   - layers.Dense(1)          # regresja: jedno wyjście, BEZ funkcji aktywacji
# Następnie skompiluj model:
#   - optimizer: tf.keras.optimizers.Adam(learning_rate=0.001)
#   - loss: "mse"
#   - metrics: [tf.keras.metrics.MeanAbsoluteError(name="mae")]

model = None  # <- zastąp tę linię

model.summary()

** Sprawdzenie**

In [ ]:
assert model is not None, "Zmienna model jest pusta – uzupełnij TODO 2."
assert any(isinstance(l, layers.Conv1D) for l in model.layers), "Model nie zawiera warstwy Conv1D."
assert model.output_shape == (None, 1), f"Wyjście powinno mieć kształt (None, 1), jest {model.output_shape}."
assert model.optimizer is not None and model.loss is not None, "Model nie jest skompilowany."
print("OK – model zbudowany i skompilowany.")

### Krok 9b. Trening

Callback `early_stop` zdefiniuj **pod tą nazwą** – używa go też model porównawczy Dense w sekcji 7.

In [ ]:
# TODO 3: uruchom trening.
# 1) Zdefiniuj callback early_stop = tf.keras.callbacks.EarlyStopping(...)
#    monitorujący "val_loss", z patience (np. 20) i restore_best_weights=True.
# 2) Wytrenuj model i zapisz wynik do zmiennej history:
#    dane: X_train_conv, y_train_scaled; validation_split=0.2;
#    epochs=200; batch_size=32; callbacks=[early_stop]; verbose=0 lub 1.

early_stop = None  # <- zastąp
history = None     # <- zastąp

**✅ Sprawdzenie**

In [ ]:
assert history is not None, "Brak obiektu history – uzupełnij TODO 3."
assert "val_loss" in history.history, "Brak val_loss – czy ustawiłeś validation_split?"
assert early_stop is not None, "Zdefiniuj early_stop – przyda się w sekcji 7."
print(f"OK – trening zakończony po {len(history.history['loss'])} epokach.")

## 5. Ewaluacja

### Krok 10. MAE i RMSE na zbiorze testowym
Predykcje wracają z przestrzeni standaryzowanej do jednostek glukozy (`inverse_transform`), więc metryki są w mg/dl.
Dla porównania liczymy też „model naiwny”, który zawsze zwraca średnią z treningu – sieć powinna być od niego lepsza.

In [ ]:
y_pred_scaled = model.predict(X_test_conv, verbose=0)
y_pred = y_scaler.inverse_transform(y_pred_scaled)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

y_naive = np.full_like(y_test, y_train.mean(), dtype=float)
mae_naive = mean_absolute_error(y_test, y_naive)
rmse_naive = np.sqrt(mean_squared_error(y_test, y_naive))

print(f"Conv1D  MAE: {mae:.2f}  RMSE: {rmse:.2f}")
print(f"Naiwny  MAE: {mae_naive:.2f}  RMSE: {rmse_naive:.2f}  (zawsze średnia z train)")

### Porównanie błędu train vs test
Duża różnica na korzyść zbioru treningowego to sygnał overfittingu.

In [ ]:
y_pred_train = y_scaler.inverse_transform(model.predict(X_train_conv, verbose=0))
mae_train = mean_absolute_error(y_train, y_pred_train)
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
print(f"Train MAE: {mae_train:.2f}  RMSE: {rmse_train:.2f}")
print(f"Test  MAE: {mae:.2f}  RMSE: {rmse:.2f}")

### Krok 11. Krzywa uczenia oraz wartości rzeczywiste kontra predykcje

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="validation loss")
plt.xlabel("Epoka")
plt.ylabel("MSE na skali standaryzowanej")
plt.legend()
plt.title("Krzywa uczenia")
plt.show()

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.xlabel("Glucose – wartość rzeczywista")
plt.ylabel("Glucose – predykcja")
plt.title("Predykcje modelu Conv1D")
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims)
plt.show()

## 6. Interpretacja

### Krok 12. TODO 4: napisz 3–5 zdań interpretacji
Odpowiedz w tej komórce (kliknij dwukrotnie, aby edytować):
- Czy model działa sensownie (porównaj z modelem naiwnym)?
- Gdzie się myli najbardziej – dla jakich wartości `Glucose` (spójrz na wykres rozrzutu)?
- Dlaczego Conv1D w tym zadaniu jest raczej demonstracją architektury niż oczywistym wyborem produkcyjnym?

**Twoja interpretacja:**

_…_

## 7. Mini-wariant porównawczy: Dense baseline

Model kontrolny bez konwolucji. Celem nie jest pobicie Conv1D za wszelką cenę, tylko sprawdzenie,
czy konwolucja daje przewagę na tak małym zbiorze. Dense przyjmuje dane 2D – używamy `X_train_scaled`, nie `X_train_conv`.

In [ ]:
dense_model = models.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1)
])
dense_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=[tf.keras.metrics.MeanAbsoluteError(name="mae")]
)
dense_history = dense_model.fit(
    X_train_scaled,
    y_train_scaled,
    validation_split=0.2,
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

y_pred_dense_scaled = dense_model.predict(X_test_scaled, verbose=0)
y_pred_dense = y_scaler.inverse_transform(y_pred_dense_scaled)
mae_dense = mean_absolute_error(y_test, y_pred_dense)
rmse_dense = np.sqrt(mean_squared_error(y_test, y_pred_dense))

print(f"Conv1D  MAE: {mae:.2f}  RMSE: {rmse:.2f}")
print(f"Dense   MAE: {mae_dense:.2f}  RMSE: {rmse_dense:.2f}")

## 8. Zadania dodatkowe dla szybszych grup
Każde zadanie ma własną komórkę z `TODO`. Możesz skopiować i zmodyfikować kod z wcześniejszych sekcji.

### D1. Zmień target z `Glucose` na `BMI`. Czy model uczy się łatwiej czy trudniej?

In [ ]:
# TODO D1: target_col = "BMI"; Glucose trafia do feature_cols. Powtórz kroki 4–10 i porównaj MAE/RMSE.
# Pamiętaj: błędy dla BMI są w innych jednostkach niż dla Glucose – porównuj je np. z modelem naiwnym.

### D2. Dodaj kolumnę `Outcome` jako cechę wejściową i sprawdź, czy wynik regresji się poprawia. Omów ryzyko interpretacyjne takiego wyboru.

In [ ]:
# TODO D2: dodaj "Outcome" do feature_cols, przetrenuj model, porównaj metryki.
# Pytanie do dyskusji: czy w realnym scenariuszu znalibyśmy Outcome PRZED pomiarem glukozy?

### D3. Porównaj Conv1D z prostą siecią Dense bez konwolucji (sekcja 7).

In [ ]:
# TODO D3: zestaw w tabeli (pd.DataFrame) MAE i RMSE: model naiwny, Conv1D, Dense.
# Uruchom oba modele dla kilku wartości SEED – czy różnica jest stabilna?

### D4. Zmień `kernel_size` z 2 na 3. Co dzieje się z MAE i RMSE?

In [ ]:
# TODO D4: zbuduj kopię modelu z kernel_size=3, wytrenuj i porównaj metryki.

### D5. Zmień kolejność cech wejściowych. Czy wynik modelu jest stabilny? Co to mówi o Conv1D na danych tabelarycznych?

In [ ]:
# TODO D5: wylosuj permutację feature_cols, np.
# rng = np.random.default_rng(0); shuffled = list(rng.permutation(feature_cols))
# Powtórz kroki 4–10 dla kilku permutacji i zanotuj MAE.

### D6. Dodaj `Dropout` lub `BatchNormalization` i porównaj krzywe uczenia.

In [ ]:
# TODO D6: dodaj layers.Dropout(0.2) lub layers.BatchNormalization() do architektury,
# wytrenuj i narysuj loss/val_loss obok wersji bazowej.

## 9. Pytania kontrolne
Odpowiedz krótko pod każdym pytaniem (kliknij dwukrotnie, aby edytować).

1. Dlaczego X dla Conv1D musi mieć trzy wymiary?
   **Odpowiedź:** …
2. Czym różni się loss MSE od metryki MAE?
   **Odpowiedź:** …
3. Dlaczego skalujemy zmienną docelową y w regresji neuronowej?
   **Odpowiedź:** …
4. Dlaczego wartości 0 w niektórych kolumnach Pima mogą oznaczać brak danych?
   **Odpowiedź:** …
5. Dlaczego Conv1D na danych tabelarycznych wymaga ostrożnej interpretacji?
   **Odpowiedź:** …
6. Co oznacza overfitting na wykresie loss/val_loss?
   **Odpowiedź:** …
7. Kiedy użyłbyś prostego modelu Dense zamiast Conv1D?
   **Odpowiedź:** …